# Shared-view / high-five MRD (Damianou & Lawrence 2013, §4.2 / figs. 4–5)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/PyDeepGP/blob/main/examples/damianou2013/notebooks/04_highfive_mrd.ipynb)

**Paper:** [Damianou, A. and Lawrence, N. D. (2013). *Deep Gaussian Processes*. In *AISTATS* (pp. 207–215).](https://proceedings.mlr.press/v31/damianou13a.html)

Two conditionally independent observation views share a parent latent layer (manifold relevance determination / shared GP-LVM style). In the paper this models two CMU MOCAP subjects performing a high-five; ARD on each view reveals shared vs private dimensions (fig. 4).

This notebook uses **synthetic correlated views** by default (reliable in Colab). Optionally try a pods MOCAP stand-in. Multi-view DeepGP requires CIP-0002 MRD support.


In [ ]:
# Install dependencies when running on Google Colab
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    pkgs = ["numpy>=1.7", "scipy>=0.12", "matplotlib", "GPy>=1.15"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/SheffieldML/PyDeepGP.git@main",
    ])
    print("Installed GPy and PyDeepGP for Colab.")
else:
    print("Local environment — ensure PyDeepGP and GPy are importable.")


In [ ]:
# Set QUICK=True for a short smoke run (fewer iterations / smaller models).
# Set QUICK=False for a paper-scale demonstration (slower).
QUICK = True
SEED = 42

USE_PODS = False  # set True to try pods CMU walk/jog feature split


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import GPy
import deepgp

np.random.seed(SEED)

def rbf_bias(q, ard=True):
    return GPy.kern.RBF(q, ARD=ard) + GPy.kern.Bias(q)

def ard_weights(kern):
    ls = np.asarray(kern.rbf.lengthscale).ravel() if hasattr(kern, "rbf") else np.asarray(kern.lengthscale).ravel()
    return 1.0 / np.maximum(ls, 1e-12)

def load_views(seed=42, use_pods=False):
    if use_pods:
        try:
            import pods
            data = pods.datasets.cmu_mocap_35_walk_jog()
            Y = np.asarray(data["Y"], dtype=float)[:80]
            mid = Y.shape[1] // 2
            print("Using pods walk/jog feature split as a two-view stand-in.")
            return Y[:, :mid], Y[:, mid:], "pods"
        except Exception as exc:
            print("pods unavailable:", exc)
    rng = np.random.RandomState(seed)
    n, q, d = 60, 3, 20
    X = rng.randn(n, q)
    Y1 = X @ rng.randn(q, d) + 0.05 * rng.randn(n, d)
    Y2 = X @ rng.randn(q, d) + 0.05 * rng.randn(n, d)
    return Y1, Y2, "synthetic"


In [ ]:
Y1, Y2, source = load_views(SEED, use_pods=USE_PODS)
Y1 = (Y1 - Y1.mean(0)) / (Y1.std() + 1e-12)
Y2 = (Y2 - Y2.mean(0)) / (Y2.std() + 1e-12)
print(f"Views {Y1.shape}, {Y2.shape} (source={source})")


## Fit shared-hierarchy DeepGP

Observation kernels are a **list of two kernels** (one per view), each with input dimension equal to the shared latent width.


In [ ]:
from GPy.core.parameterization.variational import NormalPosterior, VariationalPosterior

def out_var(y):
    if isinstance(y, (NormalPosterior, VariationalPosterior)):
        return float(np.asarray(y.mean).var())
    return float(np.asarray(y).var())

q1, q2 = (4, 3) if QUICK else (6, 4)
M = 15 if QUICK else 30
it_f, it_u = (40, 80) if QUICK else (200, 800)

m = deepgp.DeepGP(
    [[Y1.shape[1], Y2.shape[1]], q1, q2],
    [Y1, Y2],
    kernels=[[rbf_bias(q1), rbf_bias(q1)], rbf_bias(q2)],
    num_inducing=M,
    back_constraint=False,
)
for layer in m.layers:
    if hasattr(layer, "views"):
        for view in layer.views:
            view.Gaussian_noise.variance = out_var(view.Y) * 0.01
            view.Gaussian_noise.variance.fix()
    else:
        layer.Gaussian_noise.variance = out_var(layer.Y) * 0.01
        layer.Gaussian_noise.variance.fix()
m.optimize(max_iters=it_f, messages=False)
for layer in m.layers:
    if hasattr(layer, "views"):
        for view in layer.views:
            view.Gaussian_noise.variance.unfix()
    else:
        layer.Gaussian_noise.variance.unfix()
m.optimize(max_iters=it_u, messages=True)
print(f"Variational bound: {float(m.log_likelihood()):.3f}")


## ARD per view and top layer (fig. 4 style)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for i, view in enumerate(m.layers[0].views):
    w = ard_weights(view.kern)
    axes[i].bar(np.arange(1, len(w) + 1), w)
    axes[i].set_title(f"view {i} ARD")
    axes[i].set_xlabel("shared dim")
    print(f"View {i} ARD:", np.array2string(w, precision=3))
w_top = ard_weights(m.layers[-1].kern)
axes[2].bar(np.arange(1, len(w_top) + 1), w_top)
axes[2].set_title("top layer ARD")
fig.suptitle("Shared MRD ARD weights (fig. 4 style)")
fig.tight_layout()
plt.show()

# 2D projection of dominant shared dimensions from view 0's posterior mean
Xshare = np.asarray(m.layers[0].X.mean)
w0 = ard_weights(m.layers[0].views[0].kern)
dims = list(np.argsort(-w0)[:2])
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot(Xshare[:, dims[0]], Xshare[:, dims[1]], "o-", markersize=4)
ax.set_xlabel(f"shared dim {dims[0]}")
ax.set_ylabel(f"shared dim {dims[1]}")
ax.set_title("Shared latent trajectory (fig. 5 style)")
plt.show()


## References

- Damianou, A. and Lawrence, N. D. (2013). *Deep Gaussian Processes*. In *AISTATS* (pp. 207–215). — [PDF](https://proceedings.mlr.press/v31/damianou13a.html), §4.2
- Damianou et al. (2012), shared / MRD GP-LVM
- Lawrence & Moore (2007), hierarchical GP-LVM (high-five MOCAP)
- Scripts: `examples/damianou2013/highfive_shared.py`, `examples/example_mrd.py`
- CIP-0002: MRD revival on the maintained DeepGP path
